# Clarté Commerce — Exploratory Data Analysis

**Client:** Clarté Commerce S.A.S.  
**Analyst:** Nurbol Sultanov  
**Date:** February 2026  

Initial exploration of transaction, customer, and product data (Jan 2022 – Dec 2024).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

## 1. Load Data

In [ ]:
transactions = pd.read_csv('../data/raw/transactions.csv')
customers = pd.read_csv('../data/raw/customers.csv')
products = pd.read_csv('../data/raw/products.csv')

print(f'Transactions: {transactions.shape}')
print(f'Customers: {customers.shape}')
print(f'Products: {products.shape}')

Transactions: (92330, 11)
Customers: (12005, 9)
Products: (800, 7)


In [3]:
transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92330 entries, 0 to 92329
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    92330 non-null  object 
 1   customer_id       92330 non-null  object 
 2   transaction_date  92330 non-null  object 
 3   product_id        92330 non-null  object 
 4   quantity          92330 non-null  int64  
 5   unit_price        92330 non-null  float64
 6   total_amount      92330 non-null  float64
 7   channel           92330 non-null  object 
 8   store_id          92330 non-null  object 
 9   payment_method    92330 non-null  object 
 10  discount_applied  92330 non-null  float64
dtypes: float64(3), int64(1), object(7)
memory usage: 7.8+ MB


In [4]:
transactions.head(2)

,transaction_id,customer_id,transaction_date,product_id,quantity,unit_price,total_amount,channel,store_id,payment_method,discount_applied
0,TXN-2022-000115,CLR-00097,2022-01-01,SKU-0138,1,72.9,72.90,online,,apple_pay,0.00
1,TXN-2022-000261,CLR-00230,2022-01-01,SKU-0478,2,89.9,143.84,store,STR-148,carte_bancaire,0.20


## 2. Data Quality Overview

In [5]:
print('Missing values in transactions:')
print(transactions.isnull().sum())
print()
print('Missing values in customers:')
print(customers.isnull().sum())

Missing values in transactions:
transaction_id      0
customer_id         0
transaction_date    0
product_id          0
quantity            0
unit_price          0
total_amount        0
channel             0
store_id            0
payment_method      0
discount_applied    0
dtype: int64

Missing values in customers:
customer_id         0
registration_date   0
age_group           0
gender              0
city                0
region              0
preferred_channel   0
loyalty_tier        0
email_opt_in        0
dtype: int64


In [6]:
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'])
customers['registration_date'] = pd.to_datetime(customers['registration_date'])

print(f"Date range: {transactions['transaction_date'].min().date()} to {transactions['transaction_date'].max().date()}")
print(f"Unique customers: {transactions['customer_id'].nunique()}")
print(f"Unique products: {transactions['product_id'].nunique()}")
print(f"Total revenue: \u20ac{transactions['total_amount'].sum():,.2f}")

Date range: 2022-01-01 to 2024-12-31
Unique customers: 11577
Unique products: 800
Total revenue: €10,287,453.21


## 3. Transaction Volume Over Time

In [7]:
monthly = transactions.groupby(transactions['transaction_date'].dt.to_period('M')).agg(
    txn_count=('transaction_id', 'count'),
    revenue=('total_amount', 'sum'),
    unique_customers=('customer_id', 'nunique'),
    avg_order_value=('total_amount', 'mean')
).reset_index()

monthly['transaction_date'] = monthly['transaction_date'].dt.to_timestamp()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Transaction count
axes[0, 0].plot(monthly['transaction_date'], monthly['txn_count'], color='#2c3e50', linewidth=1.5)
axes[0, 0].axvline(x=pd.Timestamp('2023-08-15'), color='red', linestyle='--', alpha=0.7, label='Rebrand')
axes[0, 0].set_title('Monthly Transaction Count')
axes[0, 0].legend()

# Revenue
axes[0, 1].plot(monthly['transaction_date'], monthly['revenue'], color='#27ae60', linewidth=1.5)
axes[0, 1].axvline(x=pd.Timestamp('2023-08-15'), color='red', linestyle='--', alpha=0.7, label='Rebrand')
axes[0, 1].set_title('Monthly Revenue (\u20ac)')
axes[0, 1].legend()

# Unique customers
axes[1, 0].plot(monthly['transaction_date'], monthly['unique_customers'], color='#8e44ad', linewidth=1.5)
axes[1, 0].axvline(x=pd.Timestamp('2023-08-15'), color='red', linestyle='--', alpha=0.7, label='Rebrand')
axes[1, 0].set_title('Monthly Unique Customers')
axes[1, 0].legend()

# AOV
axes[1, 1].plot(monthly['transaction_date'], monthly['avg_order_value'], color='#e67e22', linewidth=1.5)
axes[1, 1].axvline(x=pd.Timestamp('2023-08-15'), color='red', linestyle='--', alpha=0.7, label='Rebrand')
axes[1, 1].set_title('Average Order Value (\u20ac)')
axes[1, 1].legend()

plt.suptitle('Clart\u00e9 Commerce \u2014 Key Metrics Over Time', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/monthly_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Channel Distribution

In [8]:
REBRAND_DATE = pd.Timestamp('2023-08-15')

pre = transactions[transactions['transaction_date'] < REBRAND_DATE]
post = transactions[transactions['transaction_date'] >= REBRAND_DATE]

print('Pre-rebrand channel split:')
print(pre['channel'].value_counts(normalize=True).apply(lambda x: f'{x:.1%}'))
print()
print('Post-rebrand channel split:')
print(post['channel'].value_counts(normalize=True).apply(lambda x: f'{x:.1%}'))

Pre-rebrand channel split:
channel
store     54.4%
online    45.6%

Post-rebrand channel split:
channel
online    50.3%
store     49.7%


## 5. Customer Demographics

In [9]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Age distribution
age_order = ['18-24', '25-34', '35-44', '45-54', '55+']
customers['age_group'].value_counts().reindex(age_order).plot(
    kind='bar', ax=axes[0], color='#3498db', edgecolor='white'
)
axes[0].set_title('Customer Age Distribution')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=0)

# Gender
customers['gender'].value_counts().plot(
    kind='bar', ax=axes[1], color=['#e74c3c', '#3498db', '#2ecc71'], edgecolor='white'
)
axes[1].set_title('Gender Distribution')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)

# Loyalty tiers
tier_order = ['bronze', 'silver', 'gold', 'platinum']
customers['loyalty_tier'].value_counts().reindex(tier_order).plot(
    kind='bar', ax=axes[2], color=['#cd7f32', '#c0c0c0', '#ffd700', '#e5e4e2'], edgecolor='white'
)
axes[2].set_title('Loyalty Tier Distribution')
axes[2].set_xlabel('')
axes[2].tick_params(axis='x', rotation=0)

plt.suptitle('Clart\u00e9 Commerce \u2014 Customer Demographics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/customer_demographics.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Revenue by Category

In [10]:
txn_prod = transactions.merge(products[['product_id', 'category']], on='product_id', how='left')

cat_revenue = txn_prod.groupby('category')['total_amount'].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
cat_revenue.plot(kind='barh', color='#2c3e50', edgecolor='white', ax=ax)
ax.set_title('Total Revenue by Product Category (2022\u20132024)', fontsize=13, fontweight='bold')
ax.set_xlabel('Revenue (\u20ac)')
ax.set_ylabel('')

for i, v in enumerate(cat_revenue):
    ax.text(v + 5000, i, f'\u20ac{v:,.0f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../reports/figures/revenue_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Initial Observations

- Dataset covers 3 full years (2022\u20132024), ~92K transactions across ~11.6K unique customers
- No missing values in core fields
- Clear seasonality: peaks in January (soldes d'hiver), June (soldes d'\u00e9t\u00e9), November (Black Friday)
- **Post-rebrand shift visible**: transaction volume and unique customer count drop noticeably after Aug 2023
- Channel mix shifting toward online post-rebrand (store: 54% \u2192 50%)
- AOV increased post-rebrand \u2014 consistent with the "affordable luxury" repositioning

**Next steps:** RFM segmentation to quantify which customer segments are most affected.